In [1]:
from firedrake import *
from ufl import as_vector, as_matrix
import numpy as np
import os
current_path = os.getcwd()
print(current_path)


firedrake:WARNING OMP_NUM_THREADS is not set or is set to a value greater than 1, we suggest setting OMP_NUM_THREADS=1 to improve performance


/home/galatij/firedrake_install/venv-firedrake/homogenization


In [2]:
# MESH
n = 12
mesh = PeriodicUnitCubeMesh(n, n, n, hexahedral=False)

# FUNCTION SPACES
P0 = FunctionSpace(mesh, "DG", 0)
P1 = VectorFunctionSpace(mesh, "CG", 2)
P1 = VectorFunctionSpace(mesh, "CG", 1)
W = P1 * P1

nullspace = MixedVectorSpaceBasis(
    W,
    [W.sub(0), VectorSpaceBasis(constant=True)],
)

# DATA
x, y, z = SpatialCoordinate(mesh)

mu = Function(P0).interpolate(
    4e-10 + sin(2*pi * x) + sin(2*pi * y) + sin(2*pi * z)
)

mu_top = Constant(0.2)
mu_bottom = Constant(0.8)

mu = Function(P0).interpolate(
    conditional(z > 0.5, mu_top, mu_bottom)
)

f_fun = as_vector((sin(2*pi * x*y*z), cos(2*pi * y), 0.))
# f_fun = as_vector((0., 0., -2e-10))
f_fun = as_vector((0., 0., sin(2*pi*z)))
f = Function(P1).interpolate(f_fun)

# VARIATIONAL PROBLEM
u, r = TrialFunctions(W)
u_test, r_test = TestFunctions(W)

def mskew(A):
    return as_vector([
        A[2,1] - A[1,2],
        A[0,2] - A[2,0],
        A[1,0] - A[0,1]
    ])

def vskw(v):
    return as_matrix([
        [    0, -v[2],  v[1]],
        [ v[2],     0, -v[0]],
        [-v[1],  v[0],     0]
    ])
    
a = (mu * inner(grad(u), grad(u_test)) + dot(mskew(grad(r)), u_test) + dot(mskew(grad(u)), r_test) + 2 / mu * dot(r, r_test))*dx

# # artificially kill the nullspace
# eps = Constant(1e-5)
# a += eps * dot(u, u_test) * dx

L = (dot(f, u_test)) * dx

w = Function(W)
solve(a == L, w, nullspace = nullspace)
uh, rh = w.subfunctions

# EXPORT SOLUTION
output_folder = "output/"
outfileU = VTKFile(current_path+"/displacement.pvd")
outfileR = VTKFile(current_path+"/rotation.pvd")
uh.rename("displacement")
rh.rename("rotation")

outfileU.write(uh)
outfileR.write(rh)

firedrake:WARNING No comm specified for VectorSpaceBasis, COMM_WORLD assumed
